[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/077.%20The%20Dynamic%20%26%20Real-Time%20VRP/P77-Tier-7_executed.ipynb)

---

# 77. The Dynamic and Real-Time Vehicle Routing Problem
## Tier 7: The Human–AI Symbiotic Partnership

### Purpose and scope

This notebook demonstrates a human–AI decision-support workflow for dynamic vehicle-routing requests. An explainable AI component generates a recommendation and a heuristic confidence score; a human operator reviews it, may modify it, and contributes relationship and exception-handling context. The partnership records interactions and updates a simple trust score.

The implementation is an illustrative simulation, not a vehicle-routing optimizer. It does not construct routes, reserve vehicles, enforce vehicle availability across requests, or optimize a fleet-wide objective. Travel-time estimates are simplified Euclidean-distance estimates. Confidence, trust, decision quality, and customer satisfaction are heuristic or explicitly stipulated scenario values; they are not statistically calibrated or empirical performance measurements. The benchmark values used in the approach comparison are illustrative assumptions, not results produced by this simulation.

### Inputs and outputs

Inputs include request attributes (such as origin, destination, demand, deadline, and priority), vehicle and operating context, and optional customer profiles. The workflow returns an AI recommendation, human modifications, a final collaborative decision, and interaction-level summary metrics and visualizations.

### Modeling assumptions

- Coordinates are points in a two-dimensional plane, and distance is Euclidean distance in coordinate units.
- The travel-time estimate uses two minutes per distance unit, multiplied by a random factor between 1.0 and 1.2. The scenario does not provide a calibrated road network or traffic model.
- Priority level 1 denotes the highest priority; levels increase as priority decreases.
- Confidence and quality scores are heuristic values in the interval [0, 1]. They are not probability estimates unless separately calibrated.
- Human review is represented by explicit rules. The simulation does not model operator availability, workload, or the time required for human review.
- Requests are processed independently. Vehicle assignments are recommendations and are not reserved between requests.

### Partnership workflow

1. The AI analyzes a request, produces reasoning and uncertainty factors, and proposes a recommendation with alternatives.
2. A human operator reviews the recommendation using AI confidence, expertise rules, and optional customer-relationship information.
3. Human modifications are applied to the AI recommendation to form the collaborative decision.
4. The interaction is recorded, and simple trust and learning indicators are updated.
5. A crisis scenario and a separate illustrative benchmark comparison are examined.

### Formal description of the decision flow

For request $r_t$ and operating context $c_t$, the AI produces a recommendation $a_t$ and heuristic confidence $q_t \in [0,1]$. The operator produces a modification map $m_t$. The final decision is the recommendation with those modifications applied:

$$
d_t = a_t \oplus m_t,
$$

where $\oplus$ means that values in $m_t$ replace matching keys in $a_t$, while keys not present in $m_t$ remain unchanged. In the code, this is implemented by copying the recommendation and calling `decision.update(...)`.

The operator's trust score $u_t \in [0,1]$ is updated by fixed rules: a decision with no modifications increases trust by 0.05; a decision with more than two modifications decreases it by 0.02; and an AI confidence score above 0.8 adds 0.03. The score is clipped to $[0,1]$. These are illustrative rules, not a learned or empirically validated trust model. The implementation is in `HumanOperator._update_trust`.

The workflow is sequential rather than an optimization algorithm. Its purpose is to make the roles, information flow, and interpretation of a human–AI partnership explicit; it does not solve a mathematical VRP objective or guarantee feasible routes.

In [ ]:
from dataclasses import dataclass, field
import random
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fixed seeds make the simulated heuristic variation reproducible.
np.random.seed(42)
random.seed(42)

## 1. Data structures

The data classes make the information passed between the AI, operator, and partnership explicit. For example, `AIDecision` keeps the recommendation separate from its confidence explanation and alternative options. This separation makes the workflow easier to inspect and extend without implying that the heuristic scores are calibrated probabilities.

In [ ]:
@dataclass
class DecisionLayer:
    """Represent a layer in the symbiotic intelligence architecture."""

    layer_type: str
    ai_confidence: float
    human_trust: float
    collaboration_weight: float


@dataclass
class ConfidenceLevel:
    """Store a heuristic confidence score and its explanation."""

    score: float
    reasoning: List[str]
    uncertainty_factors: List[str]
    suggested_human_input: bool


@dataclass
class CustomerProfile:
    """Store customer-priority and relationship information."""

    customer_id: str
    priority_level: int
    relationship_strength: float
    special_requirements: List[str]
    communication_preferences: Dict[str, str]


@dataclass
class AIDecision:
    """Represent an explainable AI recommendation and its alternatives."""

    decision_type: str
    recommendation: Dict[str, Any]
    confidence: ConfidenceLevel
    alternatives: List[Dict[str, Any]]
    expected_outcomes: Dict[str, float]


@dataclass
class HumanInput:
    """Represent an operator's modifications and reasoning."""

    operator_id: str
    decision_modification: Dict[str, Any]
    reasoning: str
    confidence_override: Optional[float] = None
    relationship_considerations: List[str] = field(default_factory=list)

print("Human–AI partnership data structures are ready.")

## 2. Explainable AI component

The AI component is a transparent rule-based simulation. It estimates confidence from request completeness and two simulated pattern/context scores, provides human-readable reasons and uncertainty flags, and selects a vehicle using availability and capacity fields. Its vehicle selection is deliberately simple: it returns the first available vehicle with sufficient capacity, or a fallback when none is suitable. This is a recommendation rule, not a fleet-routing or scheduling optimization.

The confidence score is an arithmetic heuristic, not a confidence interval. A confidence threshold controls whether the system suggests human input; it does not prevent a human from reviewing the recommendation.

In [ ]:
class ExplainableAISystem:
    """Generate transparent, heuristic recommendations for requests."""

    def __init__(self, confidence_threshold: float = 0.7):
        self.confidence_threshold = confidence_threshold
        self.decision_history = []
        self.learning_insights = []

    def analyze_request(self, request: Dict[str, Any], context: Dict[str, Any]) -> AIDecision:
        """Analyze one request and return a recommendation with explanations."""
        base_confidence = self._calculate_confidence(request, context)
        reasoning = self._generate_reasoning(request, context)
        uncertainty_factors = self._identify_uncertainty_factors(request, context)

        confidence = ConfidenceLevel(
            score=base_confidence,
            reasoning=reasoning,
            uncertainty_factors=uncertainty_factors,
            suggested_human_input=base_confidence < self.confidence_threshold,
        )
        recommendation = self._generate_recommendation(request, context, confidence)
        alternatives = self._generate_alternatives(request, context)
        expected_outcomes = self._predict_outcomes(recommendation, context)

        decision = AIDecision(
            decision_type=request.get("type", "unspecified"),
            recommendation=recommendation,
            confidence=confidence,
            alternatives=alternatives,
            expected_outcomes=expected_outcomes,
        )
        self.decision_history.append(decision)
        return decision

    def _calculate_confidence(self, request: Dict[str, Any], context: Dict[str, Any]) -> float:
        """Combine completeness and simulated pattern/context scores."""
        if request:
            data_completeness = sum(value is not None for value in request.values()) / len(request)
        else:
            data_completeness = 0.0

        pattern_confidence = 0.8 + 0.2 * random.random()
        context_similarity = 0.7 + 0.3 * random.random()
        overall_confidence = (data_completeness + pattern_confidence + context_similarity) / 3
        return min(1.0, max(0.0, overall_confidence))

    def _generate_reasoning(self, request: Dict[str, Any], context: Dict[str, Any]) -> List[str]:
        """Explain request attributes that influence the recommendation."""
        reasoning = []

        if "deadline" in request:
            reasoning.append(f"The request deadline is {request['deadline']} minutes.")
        if "priority" in request:
            reasoning.append(
                f"Priority level {request['priority']} is considered in the service decision."
            )
        if "origin" in request and "destination" in request:
            distance = np.linalg.norm(np.asarray(request["origin"]) - np.asarray(request["destination"]))
            reasoning.append(f"The straight-line distance is {distance:.1f} coordinate units.")
        if "demand" in request:
            reasoning.append(
                f"Demand of {request['demand']} units is checked against vehicle capacity."
            )
        if not reasoning:
            reasoning.append("Few request attributes are available to explain the recommendation.")
        return reasoning

    def _identify_uncertainty_factors(
        self, request: Dict[str, Any], context: Dict[str, Any]
    ) -> List[str]:
        """List missing or challenging request characteristics."""
        uncertainty_factors = []
        missing_fields = [key for key, value in request.items() if value is None]
        if missing_fields:
            uncertainty_factors.append(f"Missing data for: {', '.join(missing_fields)}")

        if "deadline" in request and request["deadline"] < 30:
            uncertainty_factors.append("A tight deadline increases operational uncertainty.")

        if "origin" in request and "destination" in request:
            distance = np.linalg.norm(np.asarray(request["origin"]) - np.asarray(request["destination"]))
            if distance > 50:
                uncertainty_factors.append("A long straight-line distance increases routing complexity.")
        return uncertainty_factors

    def _generate_recommendation(
        self, request: Dict[str, Any], context: Dict[str, Any], confidence: ConfidenceLevel
    ) -> Dict[str, Any]:
        """Create the primary vehicle and travel-time recommendation."""
        if "origin" in request and "destination" in request:
            best_vehicle = self._find_best_vehicle(request, context)
            estimated_time = self._estimate_travel_time(request, context)
            return {
                "vehicle_id": best_vehicle,
                "estimated_time": estimated_time,
                "route_confidence": confidence.score,
                "priority_handling": request.get("priority", 3),
            }
        return {"status": "insufficient_data", "reason": "Missing origin or destination."}

    def _find_best_vehicle(self, request: Dict[str, Any], context: Dict[str, Any]) -> str:
        """Select the first available vehicle that can carry the request."""
        vehicles = context.get("vehicles", [])
        if not vehicles:
            return "V-DEFAULT"

        available_vehicles = [vehicle for vehicle in vehicles if vehicle.get("available", False)]
        if not available_vehicles:
            return vehicles[0]["id"]

        demand = request.get("demand", 1)
        suitable_vehicles = [
            vehicle for vehicle in available_vehicles
            if vehicle.get("capacity", 0) >= demand
        ]
        if suitable_vehicles:
            return suitable_vehicles[0]["id"]
        return available_vehicles[0]["id"]

    def _estimate_travel_time(self, request: Dict[str, Any], context: Dict[str, Any]) -> float:
        """Estimate travel time from Euclidean distance using the stated assumption."""
        if "origin" not in request or "destination" not in request:
            return 30.0

        distance = np.linalg.norm(np.asarray(request["origin"]) - np.asarray(request["destination"]))
        base_time = distance * 2.0
        uncertainty_factor = 1.0 + 0.2 * random.random()
        return base_time * uncertainty_factor

    def _generate_alternatives(
        self, request: Dict[str, Any], context: Dict[str, Any]
    ) -> List[Dict[str, Any]]:
        """Create simple illustrative alternatives for operator review."""
        alternatives = []
        if "origin" in request and "destination" in request:
            vehicles = context.get("vehicles", [])
            if len(vehicles) > 1:
                best_vehicle = self._find_best_vehicle(request, context)
                alternative_vehicle = next(
                    (vehicle["id"] for vehicle in vehicles if vehicle["id"] != best_vehicle),
                    vehicles[0]["id"],
                )
                alternatives.append(
                    {
                        "vehicle_id": alternative_vehicle,
                        "estimated_time": self._estimate_travel_time(request, context) * 1.1,
                        "route_confidence": 0.6,
                        "reason": "Alternative vehicle option.",
                    }
                )

        alternatives.append(
            {
                "vehicle_id": "DELAYED",
                "estimated_time": self._estimate_travel_time(request, context) * 1.5,
                "route_confidence": 0.8,
                "reason": "Delayed service option for a different planning choice.",
            }
        )
        return alternatives

    def _predict_outcomes(
        self, recommendation: Dict[str, Any], context: Dict[str, Any]
    ) -> Dict[str, float]:
        """Return illustrative outcome scores, not validated forecasts."""
        outcomes = {}
        if "estimated_time" in recommendation:
            estimated_time = recommendation["estimated_time"]
            if estimated_time < 20:
                satisfaction = 0.9
            elif estimated_time < 40:
                satisfaction = 0.7
            else:
                satisfaction = 0.5

            outcomes["customer_satisfaction"] = satisfaction
            outcomes["delivery_probability"] = 0.85 + 0.1 * random.random()
            outcomes["cost_efficiency"] = 0.7 + 0.2 * random.random()
        return outcomes

print("Explainable AI component is ready.")

## 3. Human operator

The operator reviews the AI recommendation rather than independently solving a routing problem. The rules represent three sources of human input: customer relationship strength, low AI confidence, and expertise under urgent deadlines. Each rule creates an explicit modification or explanation, making the effect of human review inspectable.

Trust is an illustrative state variable: it summarizes prior interactions under the specified update rules. It is not a direct measurement of a person's beliefs, and a high value does not establish that a recommendation is safe or correct.

In [ ]:
class HumanOperator:
    """Review AI recommendations using expertise and relationship context."""

    def __init__(self, operator_id: str, expertise_level: float = 0.8):
        self.operator_id = operator_id
        self.expertise_level = expertise_level
        self.relationships = {}
        self.decision_history = []
        self.trust_in_ai = 0.5

    def review_ai_recommendation(
        self,
        ai_decision: AIDecision,
        request: Dict[str, Any],
        customer_profile: Optional[CustomerProfile] = None,
    ) -> HumanInput:
        """Review the recommendation and return explicit human input."""
        ai_confidence = ai_decision.confidence.score
        relationship_factor = customer_profile.relationship_strength if customer_profile else 0.0
        expertise_assessment = self._assess_with_expertise(ai_decision, request)
        modification = self._decide_modification(
            ai_decision, request, ai_confidence, relationship_factor, expertise_assessment
        )
        reasoning = self._generate_human_reasoning(ai_decision, request, modification)

        human_input = HumanInput(
            operator_id=self.operator_id,
            decision_modification=modification,
            reasoning=reasoning,
            confidence_override=self._calculate_confidence_override(
                ai_confidence, relationship_factor
            ),
            relationship_considerations=self._get_relationship_considerations(customer_profile),
        )
        self._update_trust(ai_decision, modification)
        self.decision_history.append(human_input)
        return human_input

    def _assess_with_expertise(self, ai_decision: AIDecision, request: Dict[str, Any]) -> float:
        """Score urgency, priority, and route complexity using simple rules."""
        factors = []
        if "deadline" in request:
            if request["deadline"] < 15:
                factors.append(0.9)
            elif request["deadline"] < 30:
                factors.append(0.7)
            else:
                factors.append(0.5)
        if "priority" in request:
            factors.append(request["priority"] / 5.0)
        if "origin" in request and "destination" in request:
            distance = np.linalg.norm(np.asarray(request["origin"]) - np.asarray(request["destination"]))
            if distance > 40:
                factors.append(0.8)
            elif distance > 20:
                factors.append(0.6)
            else:
                factors.append(0.4)
        return float(np.mean(factors)) if factors else 0.5

    def _decide_modification(
        self,
        ai_decision: AIDecision,
        request: Dict[str, Any],
        ai_confidence: float,
        relationship_factor: float,
        expertise_assessment: float,
    ) -> Dict[str, Any]:
        """Apply relationship, confidence, and expertise-based review rules."""
        modification = {}
        recommendation = ai_decision.recommendation

        if relationship_factor > 0.7:
            if "estimated_time" in recommendation:
                modification["estimated_time"] = recommendation["estimated_time"] * 0.9
            if "priority_handling" in recommendation:
                modification["priority_handling"] = 1

        if ai_confidence < 0.6:
            if "estimated_time" in recommendation:
                modification["estimated_time"] = recommendation["estimated_time"] * 1.2
            if ai_decision.alternatives:
                modification["alternative_chosen"] = True
                modification["alternative_reason"] = "Low AI confidence."

        if expertise_assessment > 0.8 and request.get("deadline", float("inf")) < 20:
            modification["expert_override"] = True
            modification["expert_reason"] = "An urgent deadline requires special handling."
        return modification

    def _generate_human_reasoning(
        self, ai_decision: AIDecision, request: Dict[str, Any], modification: Dict[str, Any]
    ) -> str:
        """Explain the main reasons behind the operator's modifications."""
        reasoning_parts = []
        if ai_decision.confidence.score < 0.6:
            reasoning_parts.append("AI confidence is low, so additional caution is warranted.")
        if "expert_override" in modification:
            reasoning_parts.append("Expert review flagged an urgent, complex situation.")
        if modification.get("priority_handling") == 1:
            reasoning_parts.append("Priority was raised for the customer relationship.")
        if request.get("deadline", float("inf")) < 15:
            reasoning_parts.append("The very tight deadline calls for careful review.")
        return "; ".join(reasoning_parts) if reasoning_parts else "The AI recommendation was accepted without modification."

    def _update_trust(self, ai_decision: AIDecision, modification: Dict[str, Any]) -> None:
        """Update the illustrative trust state using fixed interaction rules."""
        if len(modification) == 0:
            self.trust_in_ai = min(1.0, self.trust_in_ai + 0.05)
        elif len(modification) > 2:
            self.trust_in_ai = max(0.0, self.trust_in_ai - 0.02)
        if ai_decision.confidence.score > 0.8:
            self.trust_in_ai = min(1.0, self.trust_in_ai + 0.03)

    def _calculate_confidence_override(
        self, ai_confidence: float, relationship_factor: float
    ) -> Optional[float]:
        """Return a heuristic human confidence override when a rule applies."""
        if relationship_factor > 0.8:
            return min(1.0, ai_confidence + 0.2)
        if ai_confidence < 0.5:
            return 0.7
        return None

    def _get_relationship_considerations(
        self, customer_profile: Optional[CustomerProfile]
    ) -> List[str]:
        """Summarize customer factors that may matter to human review."""
        considerations = []
        if customer_profile:
            if customer_profile.priority_level >= 4:
                considerations.append("High-priority customer.")
            if customer_profile.relationship_strength > 0.8:
                considerations.append("Strong relationship; customer satisfaction is important.")
            if customer_profile.special_requirements:
                considerations.append(
                    f"Special requirements: {', '.join(customer_profile.special_requirements)}"
                )
        return considerations

print("Human operator component is ready.")

## 4. Symbiotic partnership

This component connects the AI and human-review stages in a fixed order: AI analysis, human review, collaborative decision formation, interaction recording, and learning-insight recording. The order matters because the human sees the AI recommendation before modifying it. The final decision preserves the AI recommendation's fields unless the operator supplies a replacement value.

In [ ]:
class SymbioticPartnership:
    """Coordinate AI analysis, human review, and interaction recording."""

    def __init__(self, ai_system: ExplainableAISystem, human_operator: HumanOperator):
        self.ai_system = ai_system
        self.human_operator = human_operator
        self.partnership_history = []
        self.trust_evolution = []
        self.performance_metrics = []
        self.learning_insights = []

    def process_request(
        self,
        request: Dict[str, Any],
        context: Dict[str, Any],
        customer_profile: Optional[CustomerProfile] = None,
    ) -> Dict[str, Any]:
        """Run one request through the full partnership workflow."""
        ai_decision = self.ai_system.analyze_request(request, context)
        trust_before = self.human_operator.trust_in_ai
        human_input = self.human_operator.review_ai_recommendation(
            ai_decision, request, customer_profile
        )
        final_decision = self._make_collaborative_decision(ai_decision, human_input)
        self._record_interaction(
            request, ai_decision, human_input, final_decision, trust_before
        )
        self._update_learning(ai_decision, human_input, final_decision)
        return final_decision

    def _make_collaborative_decision(
        self, ai_decision: AIDecision, human_input: HumanInput
    ) -> Dict[str, Any]:
        """Apply human modifications and attach collaboration metadata."""
        decision = ai_decision.recommendation.copy()
        decision.update(human_input.decision_modification)
        decision["collaboration_metadata"] = {
            "ai_confidence": ai_decision.confidence.score,
            "human_trust": self.human_operator.trust_in_ai,
            "human_modifications": len(human_input.decision_modification),
            "final_confidence": (
                human_input.confidence_override
                if human_input.confidence_override is not None
                else ai_decision.confidence.score
            ),
            "decision_source": "human-ai-collaboration",
        }
        return decision

    def _record_interaction(
        self,
        request: Dict[str, Any],
        ai_decision: AIDecision,
        human_input: HumanInput,
        final_decision: Dict[str, Any],
        trust_before: float,
    ) -> None:
        """Store the pre-review trust and resulting decision indicators."""
        interaction = {
            "request_id": request.get("id", "unknown"),
            "ai_confidence": ai_decision.confidence.score,
            "human_trust_before": trust_before,
            "human_trust_after": self.human_operator.trust_in_ai,
            "human_modifications": len(human_input.decision_modification),
            "final_confidence": final_decision["collaboration_metadata"]["final_confidence"],
            "decision_quality": self._evaluate_decision_quality(final_decision, request),
        }
        self.partnership_history.append(interaction)
        self.trust_evolution.append(
            {"interaction": len(self.partnership_history), "trust_level": self.human_operator.trust_in_ai}
        )

    def _evaluate_decision_quality(
        self, decision: Dict[str, Any], request: Dict[str, Any]
    ) -> float:
        """Calculate a transparent heuristic score, not an observed outcome."""
        quality_score = 0.5
        if "estimated_time" in decision and "deadline" in request:
            if decision["estimated_time"] <= request["deadline"]:
                quality_score += 0.3
            else:
                quality_score -= 0.2
        confidence = decision["collaboration_metadata"]["final_confidence"]
        quality_score += 0.2 * confidence
        return min(1.0, max(0.0, quality_score))

    def _update_learning(
        self,
        ai_decision: AIDecision,
        human_input: HumanInput,
        final_decision: Dict[str, Any],
    ) -> None:
        """Record a short rule-based learning observation for later inspection."""
        insight = {
            "interaction": len(self.partnership_history),
            "ai_confidence": ai_decision.confidence.score,
            "human_trust": self.human_operator.trust_in_ai,
            "modifications_made": bool(human_input.decision_modification),
            "collaboration_success": (
                final_decision["collaboration_metadata"]["final_confidence"] > 0.7
            ),
            "learning_point": self._extract_learning_point(ai_decision, human_input),
        }
        self.learning_insights.append(insight)

    def _extract_learning_point(
        self, ai_decision: AIDecision, human_input: HumanInput
    ) -> str:
        """Classify the interaction using simple explanatory rules."""
        if ai_decision.confidence.score < 0.5 and human_input.decision_modification:
            return "Human intervention accompanied a low-confidence AI recommendation."
        if ai_decision.confidence.score > 0.8 and not human_input.decision_modification:
            return "A high-confidence AI recommendation was accepted without modification."
        if human_input.relationship_considerations:
            return "Customer relationship considerations were available to human review."
        return "Standard human–AI collaboration pattern."

    def get_partnership_metrics(self) -> Dict[str, Any]:
        """Summarize observed simulation scores and trust-state changes."""
        if not self.partnership_history:
            return {"status": "no_data"}

        history = self.partnership_history
        initial_trust = history[0]["human_trust_before"]
        return {
            "total_interactions": len(history),
            "average_ai_confidence": float(np.mean([item["ai_confidence"] for item in history])),
            "average_human_trust": float(np.mean([item["human_trust_before"] for item in history])),
            "average_final_confidence": float(np.mean([item["final_confidence"] for item in history])),
            "average_modifications_per_decision": float(
                np.mean([item["human_modifications"] for item in history])
            ),
            "average_decision_quality": float(
                np.mean([item["decision_quality"] for item in history])
            ),
            "trust_evolution": self.human_operator.trust_in_ai - initial_trust,
            "current_trust_level": self.human_operator.trust_in_ai,
            "learning_insights_count": len(self.learning_insights),
        }

print("Symbiotic partnership workflow is ready.")

## 5. Scenario data

The crisis requests, vehicle attributes, and customer profiles below define the illustrative case. They are inputs to the rule-based workflow, not evidence that the resulting recommendations are operationally feasible. For example, requests are processed independently, so the same vehicle may be recommended more than once.

In [ ]:
def create_crisis_scenario() -> Dict[str, Any]:
    """Create a fixed crisis scenario for the partnership simulation."""
    crisis_requests = [
        {
            "id": "CRISIS-001",
            "type": "emergency_delivery",
            "origin": (10, 15),
            "destination": (80, 85),
            "demand": 2,
            "deadline": 10,
            "priority": 1,
            "special_handling": "medical_supplies",
        },
        {
            "id": "CRISIS-002",
            "type": "vip_customer",
            "origin": (25, 30),
            "destination": (60, 65),
            "demand": 1,
            "deadline": 15,
            "priority": 1,
            "special_handling": "executive_transport",
        },
        {
            "id": "CRISIS-003",
            "type": "system_failure",
            "origin": (5, 10),
            "destination": (90, 95),
            "demand": 3,
            "deadline": 20,
            "priority": 2,
            "special_handling": "critical_replacement",
        },
    ]
    context = {
        "vehicles": [
            {"id": "V-EMERGENCY-1", "available": True, "capacity": 5, "special_equipment": True},
            {"id": "V-EMERGENCY-2", "available": True, "capacity": 3, "special_equipment": False},
            {"id": "V-STANDARD-1", "available": False, "capacity": 4, "special_equipment": False},
            {"id": "V-STANDARD-2", "available": True, "capacity": 2, "special_equipment": False},
        ],
        "system_status": "degraded",
        "weather_conditions": "poor",
        "traffic_conditions": "heavy",
    }
    customer_profiles = {
        "CRISIS-001": CustomerProfile(
            customer_id="HOSPITAL-001",
            priority_level=5,
            relationship_strength=0.9,
            special_requirements=["temperature_controlled", "priority_handling"],
            communication_preferences={"method": "direct_call", "frequency": "continuous_updates"},
        ),
        "CRISIS-002": CustomerProfile(
            customer_id="EXECUTIVE-001",
            priority_level=5,
            relationship_strength=0.8,
            special_requirements=["discreet_service", "premium_vehicle"],
            communication_preferences={"method": "text_message", "frequency": "milestone_updates"},
        ),
        "CRISIS-003": CustomerProfile(
            customer_id="FACILITY-001",
            priority_level=4,
            relationship_strength=0.7,
            special_requirements=["heavy_equipment", "loading_assistance"],
            communication_preferences={"method": "email", "frequency": "status_updates"},
        ),
    }
    return {
        "requests": crisis_requests,
        "context": context,
        "customer_profiles": customer_profiles,
    }


def setup_customer_profiles() -> Dict[str, CustomerProfile]:
    """Create example profiles for high-value, regular, and new customers."""
    return {
        "VIP-001": CustomerProfile(
            customer_id="CORPORATE-001",
            priority_level=5,
            relationship_strength=0.9,
            special_requirements=["executive_service", "real_time_tracking"],
            communication_preferences={"method": "direct_line", "frequency": "continuous"},
        ),
        "REG-001": CustomerProfile(
            customer_id="BUSINESS-001",
            priority_level=3,
            relationship_strength=0.6,
            special_requirements=["standard_service"],
            communication_preferences={"method": "email", "frequency": "daily"},
        ),
        "NEW-001": CustomerProfile(
            customer_id="STARTUP-001",
            priority_level=2,
            relationship_strength=0.3,
            special_requirements=["cost_sensitive"],
            communication_preferences={"method": "portal", "frequency": "milestone"},
        ),
    }

print("Crisis scenario and customer profiles are defined.")

## 6. Crisis simulation and illustrative comparison

The crisis simulation runs each request through the partnership and reports estimated travel time in minutes, confidence scores, modification counts, heuristic quality, and trust changes. These outputs describe the behavior of the stated rules; they do not establish delivery success.

The comparison function uses the same crisis simulation results for the collaborative confidence and quality indicators. The response-time and satisfaction values for the three approaches are stipulated illustrative benchmarks: AI-only (2.3 seconds and 72%), human-only (120 seconds and 83%), and collaboration (32.3 seconds and 96%). The values are not measured by the request-processing code. Response time is kept in seconds and travel-time estimates remain in minutes, so the two quantities are not conflated.

In [ ]:
def simulate_crisis_management() -> Dict[str, Any]:
    """Run the fixed crisis requests through a human–AI partnership."""
    scenario = create_crisis_scenario()
    ai_system = ExplainableAISystem(confidence_threshold=0.6)
    human_operator = HumanOperator(operator_id="CRISIS-MANAGER", expertise_level=0.9)
    partnership = SymbioticPartnership(ai_system, human_operator)
    crisis_results = []

    print("CRISIS MANAGEMENT SIMULATION")
    print("=" * 32)
    print(f"Requests: {len(scenario['requests'])}")
    print(
        "Available vehicles: "
        f"{sum(vehicle['available'] for vehicle in scenario['context']['vehicles'])}/"
        f"{len(scenario['context']['vehicles'])}"
    )
    print(
        f"Operating context: {scenario['context']['system_status']} system, "
        f"{scenario['context']['weather_conditions']} weather, "
        f"{scenario['context']['traffic_conditions']} traffic."
    )

    for request in scenario["requests"]:
        customer_profile = scenario["customer_profiles"].get(request["id"])
        decision = partnership.process_request(
            request, scenario["context"], customer_profile
        )
        metadata = decision["collaboration_metadata"]
        result = {
            "request_id": request["id"],
            "decision": decision,
            "processing_time": decision.get("estimated_time", 0.0),
            "confidence": metadata["final_confidence"],
            "modifications": metadata["human_modifications"],
            "customer_priority": customer_profile.priority_level if customer_profile else 3,
        }
        crisis_results.append(result)
        print(f"\nRequest: {request['id']} ({request['type']})")
        print(f"  Priority: {request['priority']} (1 is highest)")
        print(f"  Deadline: {request['deadline']} minutes")
        print(f"  Recommended vehicle: {decision.get('vehicle_id', 'Not available')}")
        print(f"  Estimated travel time: {result['processing_time']:.1f} minutes")
        print(f"  Final heuristic confidence: {result['confidence']:.1%}")
        print(f"  Human modifications: {result['modifications']}")

    metrics = partnership.get_partnership_metrics()
    print("\nSimulation summary:")
    print(f"  Requests processed: {len(crisis_results)}")
    print(f"  Mean heuristic decision quality: {metrics['average_decision_quality']:.1%}")
    print(f"  Mean modifications per decision: {metrics['average_modifications_per_decision']:.1f}")
    print(f"  Trust-state change: {metrics['trust_evolution']:+.3f}")
    print(f"  Learning observations: {metrics['learning_insights_count']}")

    return {
        "crisis_results": crisis_results,
        "partnership_metrics": metrics,
        "trust_evolution": partnership.trust_evolution,
        "learning_insights": partnership.learning_insights,
    }


def compare_approaches(crisis_results: Dict[str, Any]) -> Dict[str, Any]:
    """Compare illustrative benchmarks with confidence from the crisis run."""
    collaborative = crisis_results["crisis_results"]
    metrics = crisis_results["partnership_metrics"]
    comparison = {
        "ai_only": {
            "avg_processing_time": 2.3,
            "avg_confidence": float(np.mean([item["confidence"] for item in collaborative])),
            "avg_satisfaction": 0.72,
        },
        "human_only": {
            "avg_processing_time": 120.0,
            "avg_confidence": 0.85,
            "avg_satisfaction": 0.83,
        },
        "collaborative": {
            "avg_processing_time": 32.3,
            "avg_confidence": metrics["average_final_confidence"],
            "avg_satisfaction": 0.96,
        },
        "benchmark_note": (
            "Processing-time and satisfaction figures are illustrative assumptions; "
            "they are not estimated by this simulation."
        ),
    }
    print("\nIllustrative approach comparison")
    print("=" * 36)
    for approach in ("ai_only", "human_only", "collaborative"):
        values = comparison[approach]
        print(
            f"{approach.replace('_', '-').title()}: "
            f"{values['avg_processing_time']:.1f} seconds, "
            f"{values['avg_satisfaction']:.1%} stipulated satisfaction, "
            f"{values['avg_confidence']:.1%} confidence indicator"
        )
    print(comparison["benchmark_note"])
    return comparison

print("Crisis simulation and comparison functions are ready.")

## 7. Visualizing the results

The plots summarize the simulated trust state, illustrative approach benchmarks, request travel-time estimates, and recorded learning categories. They should be interpreted according to the units and assumptions: the request plot shows estimated travel time in minutes, while the approach comparison shows stipulated response times in seconds converted to a relative speed score for visualization.

In [ ]:
def visualize_symbiotic_results(
    crisis_results: Dict[str, Any], comparison_results: Dict[str, Any]
) -> None:
    """Plot trust, illustrative approach metrics, requests, and learning records."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Human–AI Partnership: Crisis Simulation", fontsize=16, fontweight="bold")

    ax1 = axes[0, 0]
    trust_data = crisis_results["trust_evolution"]
    ax1.set_title("Trust State by Interaction")
    ax1.set_xlabel("Interaction number")
    ax1.set_ylabel("Heuristic trust score")
    ax1.set_ylim(0, 1)
    ax1.set_yticks(np.linspace(0, 1, 6))
    if trust_data:
        interactions = [item["interaction"] for item in trust_data]
        trust_levels = [item["trust_level"] for item in trust_data]
        ax1.plot(interactions, trust_levels, color="tab:blue", marker="o", label="Trust state")
        ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    approaches = ["AI-only", "Human-only", "Collaborative"]
    keys = ["ai_only", "human_only", "collaborative"]
    values = [comparison_results[key] for key in keys]
    maximum_time = max(item["avg_processing_time"] for item in values)
    speed_scores = [
        (maximum_time - item["avg_processing_time"]) / maximum_time * 100
        for item in values
    ]
    confidence_scores = [item["avg_confidence"] * 100 for item in values]
    satisfaction_scores = [item["avg_satisfaction"] * 100 for item in values]
    x = np.arange(len(approaches))
    width = 0.25
    ax2.bar(x - width, speed_scores, width, label="Relative response-speed score (%)")
    ax2.bar(x, confidence_scores, width, label="Confidence indicator (%)")
    ax2.bar(x + width, satisfaction_scores, width, label="Stipulated satisfaction (%)")
    ax2.set_title("Approach Indicators (Illustrative Benchmarks)")
    ax2.set_xlabel("Approach")
    ax2.set_ylabel("Score (%)")
    ax2.set_xticks(x)
    ax2.set_xticklabels(approaches)
    ax2.set_ylim(0, 110)
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3, axis="y")

    ax3 = axes[1, 0]
    request_data = crisis_results["crisis_results"]
    priorities = [item["customer_priority"] for item in request_data]
    travel_times = [item["processing_time"] for item in request_data]
    confidences = [item["confidence"] * 100 for item in request_data]
    scatter = ax3.scatter(
        priorities, travel_times, c=confidences, s=100, alpha=0.8, cmap="viridis"
    )
    ax3.set_title("Request Estimates")
    ax3.set_xlabel("Customer priority level (1–5)")
    ax3.set_ylabel("Estimated travel time (minutes)")
    colorbar = fig.colorbar(scatter, ax=ax3)
    colorbar.set_label("Heuristic confidence (%)")
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    insight_types = {}
    for insight in crisis_results["learning_insights"]:
        label = insight["learning_point"]
        insight_types[label] = insight_types.get(label, 0) + 1
    ax4.set_title("Recorded Learning Categories")
    ax4.set_xlabel("Learning category")
    ax4.set_ylabel("Number of interactions")
    if insight_types:
        labels = list(insight_types)
        counts = [insight_types[label] for label in labels]
        short_labels = [label[:32] + "…" if len(label) > 32 else label for label in labels]
        bars = ax4.bar(short_labels, counts, alpha=0.8)
        ax4.tick_params(axis="x", labelrotation=35)
        for tick in ax4.get_xticklabels():
            tick.set_horizontalalignment("right")
        for bar, count in zip(bars, counts):
            ax4.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.05,
                str(count),
                ha="center",
                va="bottom",
            )
    ax4.grid(True, alpha=0.3, axis="y")

    fig.tight_layout()
    plt.show()

print("Visualization function is ready.")

## 8. Run the simulation

This cell runs the crisis scenario once, compares the results with the stated illustrative benchmarks, and creates the visual summary. The printed decision-quality and confidence values are heuristic outputs from the rules; benchmark response-time and satisfaction values are assumptions supplied to support a conceptual comparison.

In [ ]:
print("Starting the human–AI partnership simulation")
crisis_results = simulate_crisis_management()
comparison_results = compare_approaches(crisis_results)

metrics = crisis_results["partnership_metrics"]
print("\nPartnership summary")
print("=" * 24)
print(f"Crisis requests handled: {len(crisis_results['crisis_results'])}")
print(f"Mean heuristic decision quality: {metrics['average_decision_quality']:.1%}")
print(f"Trust-state change: {metrics['trust_evolution']:+.3f}")
print(f"Learning observations: {metrics['learning_insights_count']}")
print(
    "Illustrative collaborative satisfaction assumption: "
    f"{comparison_results['collaborative']['avg_satisfaction']:.1%}"
)
print(
    "Illustrative collaborative response-time assumption: "
    f"{comparison_results['collaborative']['avg_processing_time']:.1f} seconds"
)

visualize_symbiotic_results(crisis_results, comparison_results)

## 9. Interpretation and limitations

The central lesson is the information flow: the AI makes a recommendation with reasons and uncertainty indicators, and the operator can modify that recommendation using explicit rules and customer context. The recorded trust state and learning categories make interaction history visible, but they do not demonstrate that trust or decision quality improves in real operations.

The crisis requests have very tight deadlines relative to the simple Euclidean travel-time estimates. If estimated travel time exceeds a deadline, the quality heuristic applies a penalty; the workflow does not make an infeasible delivery feasible. Similarly, a vehicle recommendation does not guarantee capacity suitability when no available vehicle meets demand, and repeated recommendations do not account for fleet-wide assignment conflicts.

The approach comparison should be read as a conceptual illustration, not a controlled experiment. Only the collaborative confidence indicator is derived from the simulated crisis run. The response-time and satisfaction figures for the three approaches are stipulated values, and the human-only confidence value is also an assumption. A real assessment would require operational data, clearly defined outcome measures, uncertainty calibration, and an evaluation design that accounts for differences between requests and operators.

### When this partnership pattern may be useful

- High-value customer relationships where service context matters.
- Emergency response and exception handling that require human judgment.
- Business-critical decisions where recommendations should be inspectable.
- Operations where human oversight, explanation, and feedback are explicit requirements.

### Natural extensions within this modeling approach

The existing workflow could be evaluated with richer request and vehicle data, more realistic travel-time estimates, validated confidence calibration, and carefully defined operational outcome measures. Such extensions would require additional assumptions and data; they are not implemented in this notebook.

In [ ]:
def symbiotic_partnership_summary(
    crisis_results: Dict[str, Any], comparison_results: Dict[str, Any]
) -> None:
    """Print a concise interpretation based on this simulation and its assumptions."""
    metrics = crisis_results["partnership_metrics"]
    print("\n" + "=" * 78)
    print("HUMAN–AI SYMBIOTIC PARTNERSHIP SUMMARY")
    print("=" * 78)
    print("\nPartnership concepts")
    print("  • AI contributes fast, structured analysis and explicit recommendation reasoning.")
    print("  • Human review contributes relationship context and exception handling.")
    print("  • Trust and learning indicators summarize interactions under illustrative rules.")
    print("\nSimulation indicators")
    print(f"  • Requests processed: {metrics['total_interactions']}")
    print(f"  • Mean AI heuristic confidence: {metrics['average_ai_confidence']:.1%}")
    print(f"  • Mean final heuristic confidence: {metrics['average_final_confidence']:.1%}")
    print(f"  • Mean heuristic decision quality: {metrics['average_decision_quality']:.1%}")
    print(f"  • Mean human modifications per request: {metrics['average_modifications_per_decision']:.1f}")
    print(f"  • Trust-state change: {metrics['trust_evolution']:+.3f}")
    print(f"  • Recorded learning observations: {metrics['learning_insights_count']}")
    print("\nIllustrative benchmark assumptions")
    for approach in ("ai_only", "human_only", "collaborative"):
        values = comparison_results[approach]
        label = approach.replace("_", "-").title()
        print(
            f"  • {label}: {values['avg_processing_time']:.1f} seconds; "
            f"{values['avg_satisfaction']:.1%} stipulated satisfaction"
        )
    print(f"  • {comparison_results['benchmark_note']}")
    print("\nScope")
    print("  • This is a decision-support simulation, not a fleet-routing optimizer.")
    print("  • Scores and benchmarks are heuristic or stipulated, not empirical findings.")
    print("=" * 78)

symbiotic_partnership_summary(crisis_results, comparison_results)